In [ ]:
# comment this out if running in colab
# from google.colab import drive

# drive.mount("/content/drive")
# %cd /content/drive/MyDrive/chirpy
# !pip install .

In [ ]:
#!/usr/bin/env python3
"""
Test script to compare runtime and accuracy of original vs optimized HelmholtzOperator.

This script:
1. Creates synthetic test data matching the user's use case
2. Runs both original and optimized versions
3. Times the operations
4. Asserts outputs are numerically identical (within tolerance)
5. Reports speedup metrics
"""

from __future__ import annotations
import os
import sys
import time
from pathlib import Path
import numpy as np

# -----------------------------------------------------------------------------
# Configurable working directory (no notebook magic)
# -----------------------------------------------------------------------------
WORKDIR = Path("/content/drive/MyDrive/chirpy/examples")
if WORKDIR.exists():
    os.chdir(WORKDIR)

# -----------------------------------------------------------------------------
# GPU / CuPy check
# -----------------------------------------------------------------------------
try:
    import cupy as cp

    GPU_AVAILABLE = True
    print("✓ CuPy available - GPU tests enabled")
except Exception as e:
    GPU_AVAILABLE = False
    print("✗ CuPy not available - GPU tests disabled")
    print("  Error:", e)
    sys.exit(1)

# -----------------------------------------------------------------------------
# Imports from chirpy (original) and your optimized modules
# -----------------------------------------------------------------------------
print("\nImporting modules...")
try:
    from chirpy.geometry import ImageGrid2D, TransducerArray2D
    from chirpy.data import AcquisitionData
    from chirpy.optimization.operator.helmholtz import HelmholtzOperator
    from chirpy.optimization.operator.helmholtz_optimized import (
        HelmholtzOperator_Optimized,
    )

    print("✓ Original chirpy modules imported")
except Exception as e:
    print(f"✗ Failed to import original modules: {e}")
    print("  This test needs the original chirpy library installed")
    sys.exit(1)


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def to_numpy(x):
    """Return a NumPy array regardless of whether x is NumPy or CuPy."""
    if isinstance(x, np.ndarray):
        return x
    try:
        return cp.asnumpy(x)
    except Exception:
        return np.asarray(x)


def get_tx_xy(tx_array, grid: ImageGrid2D) -> np.ndarray:  # noqa
    """
    Return transmitter (x,y) positions as a (N,2) float64 array in the same units as grid (meters).
    Tries several attribute conventions; falls back to mapping stored grid indices.
    """
    # 1) Direct (N,2) or (2,N) arrays of positions
    for name in ("positions", "xy", "coords", "centers", "elements_xy"):
        if hasattr(tx_array, name):
            arr = np.asarray(getattr(tx_array, name))
            if arr.ndim == 2 and arr.shape[1] == 2:
                return arr.astype(np.float64, copy=False)
            if arr.ndim == 2 and arr.shape[0] == 2:
                return arr.T.astype(np.float64, copy=False)

    # 2) Separate x/y arrays
    for xn, yn in (
        ("x", "y"),
        ("xs", "ys"),
        ("x_pos", "y_pos"),
        ("x_coords", "y_coords"),
    ):
        if hasattr(tx_array, xn) and hasattr(tx_array, yn):
            x = np.asarray(getattr(tx_array, xn)).astype(np.float64, copy=False)
            y = np.asarray(getattr(tx_array, yn)).astype(np.float64, copy=False)
            if x.ndim == 1 and y.ndim == 1 and x.size == y.size:
                return np.stack([x, y], axis=1)

    # 3) Grid indices → physical coords via ImageGrid2D
    for name in (
        "indices",
        "grid_indices",
        "element_indices",
        "ij",
        "yx_idx",
        "xy_idx",
    ):
        if hasattr(tx_array, name):
            idx = np.asarray(getattr(tx_array, name))
            # Normalize to shape (N,2)
            candidates = []
            if idx.ndim == 2 and idx.shape[1] == 2:
                candidates = [
                    ("iy_ix", (idx[:, 0], idx[:, 1])),
                    ("ix_iy", (idx[:, 1], idx[:, 0])),
                ]
            elif idx.ndim == 2 and idx.shape[0] == 2:
                candidates = [
                    ("iy_ix", (idx[0, :], idx[1, :])),
                    ("ix_iy", (idx[1, :], idx[0, :])),
                ]

            for _, (a, b) in candidates:
                # Try interpreting as (iy, ix)
                if (
                    a.min() >= 0
                    and b.min() >= 0
                    and a.max() < grid.ny
                    and b.max() < grid.nx
                ):
                    x = grid.xi[b]  # ix → x
                    y = grid.yi[a]  # iy → y
                    return np.stack([x, y], axis=1).astype(np.float64, copy=False)

    # 4) Nothing matched: show available attributes
    raise AttributeError(
        "Could not find transmitter coordinates on TransducerArray2D. "
        f"Available attributes: {sorted(attr for attr in dir(tx_array) if not attr.startswith('_'))}"
    )


def nearest_idx(axis_vals: np.ndarray, coords: np.ndarray) -> np.ndarray:
    """Nearest index mapping for monotone axis_vals."""
    j = np.searchsorted(axis_vals, coords)
    j = np.clip(j, 1, len(axis_vals) - 1)
    left = axis_vals[j - 1]
    right = axis_vals[j]
    return np.where(np.abs(coords - left) <= np.abs(coords - right), j - 1, j).astype(
        np.int32
    )


# -----------------------------------------------------------------------------
# Test Configuration (matching your script)
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TEST CONFIGURATION")
print("=" * 70)

PAD_TO = 180
DX = DY = 1.0e-3  # 1 mm spacing

N_TX = 368
RADIUS_M = 70e-3
C0_BG = 1500.0

F_SOS = np.arange(0.25, 0.35, 0.02) * 1e6  # 5 frequencies
FREQS = F_SOS
N_FREQ_TEST = len(FREQS)

PML_ALPHA = 10.0
PML_SIZE = 9.0e-3
SIGN_CONV = -1

print(f"Grid size: {PAD_TO}x{PAD_TO}")
print(f"Grid spacing: {DX*1e3:.1f} mm")
print(f"Number of transmitters: {N_TX}")
print(f"Test frequencies: {N_FREQ_TEST}")
print(f"Frequency range: {FREQS[0]/1e6:.2f} - {FREQS[-1]/1e6:.2f} MHz")

# -----------------------------------------------------------------------------
# Create Synthetic Test Data
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("CREATING SYNTHETIC TEST DATA")
print("=" * 70)

grid_pad = ImageGrid2D(nx=PAD_TO, ny=PAD_TO, dx=DX, dy=DY)
print(f"✓ Created ImageGrid2D: {grid_pad.shape}")

tx_array = TransducerArray2D.from_ring_array_2D(r=RADIUS_M, grid=grid_pad, n=N_TX)
print(f"✓ Created TransducerArray2D: {N_TX} elements")

# Heterogeneous c(x,y)
c_true = np.ones((PAD_TO, PAD_TO), dtype=np.float64) * C0_BG
y, x = np.ogrid[:PAD_TO, :PAD_TO]
center = PAD_TO // 2
for offset in [(-20, -20), (20, 20), (-20, 20), (20, -20)]:
    mask = (x - (center + offset[0])) ** 2 + (y - (center + offset[1])) ** 2 < 15**2
    c_true[mask] = 1540.0
print(
    f"✓ Created heterogeneous sound speed map (range: {c_true.min():.0f} - {c_true.max():.0f} m/s)"
)

# AcquisitionData with frequencies (dummy array)
dummy_array = np.zeros((N_TX, N_TX, N_FREQ_TEST), dtype=np.complex128)
acq_data = AcquisitionData(
    array=dummy_array,
    tx_array=tx_array,
    grid=grid_pad,
    freqs=FREQS,
    c0=C0_BG,
)

# Build context (no reliance on tx_array.pos)
elem_mask = np.ones((N_TX, N_TX), dtype=bool)
tx_xy = get_tx_xy(tx_array, grid_pad)  # (N_TX, 2)

x_idx = nearest_idx(grid_pad.xi, tx_xy[:, 0])
y_idx = nearest_idx(grid_pad.yi, tx_xy[:, 1])
grid_lin_idx = np.arange(
    N_TX, dtype=np.int32
)  # placeholder; adjust if your operator expects a specific mapping

acq_data.ctx = {
    "elem_mask": elem_mask,
    "x_idx": x_idx,
    "y_idx": y_idx,
    "grid_lin_idx": grid_lin_idx,
}
print("✓ Created AcquisitionData with context (mapped tx → grid indices)")


# -----------------------------------------------------------------------------
# Test function: Original vs Optimized
# -----------------------------------------------------------------------------
def test_helmholtz_operator(f_idx: int, use_optimized: bool = False):
    """Run a single frequency with either original or optimized operator; return timings and fields."""
    operator_class = HelmholtzOperator_Optimized if use_optimized else HelmholtzOperator

    # Initialize operator
    t_init_start = time.time()
    op = operator_class(
        acq_data,
        f_idx,
        sign_conv=SIGN_CONV,
        pml_alpha=PML_ALPHA,
        pml_size=PML_SIZE,
        use_gpu=GPU_AVAILABLE,
    )
    t_init = time.time() - t_init_start

    # Incident fields (homogeneous)
    t_inc_start = time.time()
    slow_inc = np.full((grid_pad.ny, grid_pad.nx), 1.0 / C0_BG, dtype=np.complex128)
    _ = op.forward(slow_inc)
    incident_fields = to_numpy(op._cache.WF.copy())
    t_inc = time.time() - t_inc_start

    # Total fields (heterogeneous)
    t_total_start = time.time()
    slow_het = (1.0 / c_true).astype(np.complex128)
    _ = op.forward(slow_het)
    total_fields = to_numpy(op._cache.WF.copy())
    t_total = time.time() - t_total_start

    scattered_fields = total_fields - incident_fields
    total_time = t_init + t_inc + t_total

    return {
        "total_time": total_time,
        "t_init": t_init,
        "t_incident": t_inc,
        "t_total": t_total,
        "incident_fields": incident_fields,
        "scattered_fields": scattered_fields,
    }


# -----------------------------------------------------------------------------
# Benchmarks
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("RUNNING BENCHMARKS")
print("=" * 70)

# Warmup GPU
if GPU_AVAILABLE:
    print("\nWarming up GPU...")
    _ = cp.zeros((100, 100))
    cp.cuda.Stream.null.synchronize()
    print("✓ GPU warmed up")

results_original = []
results_optimized = []

print(f"\nTesting {N_FREQ_TEST} frequencies...")
print("-" * 70)
for f_idx in range(N_FREQ_TEST):
    freq_mhz = FREQS[f_idx] / 1e6
    print(f"\nFrequency {f_idx+1}/{N_FREQ_TEST}: {freq_mhz:.3f} MHz")

    print("  Running ORIGINAL operator...", end=" ", flush=True)
    result_orig = test_helmholtz_operator(f_idx, use_optimized=False)
    results_original.append(result_orig)
    print(f"✓ {result_orig['total_time']:.3f}s")

    print("  Running OPTIMIZED operator...", end=" ", flush=True)
    result_opt = test_helmholtz_operator(f_idx, use_optimized=True)
    results_optimized.append(result_opt)
    print(f"✓ {result_opt['total_time']:.3f}s")

    inc_diff = np.max(
        np.abs(result_orig["incident_fields"] - result_opt["incident_fields"])
    )
    scat_diff = np.max(
        np.abs(result_orig["scattered_fields"] - result_opt["scattered_fields"])
    )
    print(f"  Max diff (incident):  {inc_diff:.2e}")
    print(f"  Max diff (scattered): {scat_diff:.2e}")

# -----------------------------------------------------------------------------
# Numerical accuracy verification
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("NUMERICAL ACCURACY VERIFICATION")
print("=" * 70)

all_passed = True
tolerance = 1e-6  # relative tolerance

for f_idx in range(N_FREQ_TEST):
    freq_mhz = FREQS[f_idx] / 1e6
    orig = results_original[f_idx]
    opt = results_optimized[f_idx]

    inc_orig = orig["incident_fields"]
    inc_opt = opt["incident_fields"]
    scat_orig = orig["scattered_fields"]
    scat_opt = opt["scattered_fields"]

    inc_rel_err = np.max(np.abs(inc_orig - inc_opt)) / (
        np.max(np.abs(inc_orig)) + 1e-15
    )
    scat_rel_err = np.max(np.abs(scat_orig - scat_opt)) / (
        np.max(np.abs(scat_orig)) + 1e-15
    )

    passed_inc = inc_rel_err < tolerance
    passed_scat = scat_rel_err < tolerance
    passed = passed_inc and passed_scat
    status = "✓ PASS" if passed else "✗ FAIL"

    print(f"\nFreq {freq_mhz:.3f} MHz: {status}")
    print(
        f"  Incident fields  - Rel. error: {inc_rel_err:.2e} {'✓' if passed_inc else '✗'}"
    )
    print(
        f"  Scattered fields - Rel. error: {scat_rel_err:.2e} {'✓' if passed_scat else '✗'}"
    )

    if not passed:
        all_passed = False
        print(f"  WARNING: Results differ beyond tolerance ({tolerance:.0e})")

print("\n" + "=" * 70)
print("✓ ALL ACCURACY TESTS PASSED" if all_passed else "✗ SOME ACCURACY TESTS FAILED")
print("=" * 70)

# -----------------------------------------------------------------------------
# Performance summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PERFORMANCE SUMMARY")
print("=" * 70)

times_orig = [r["total_time"] for r in results_original]
times_opt = [r["total_time"] for r in results_optimized]

total_orig = float(np.sum(times_orig))
total_opt = float(np.sum(times_opt))
mean_orig = float(np.mean(times_orig))
mean_opt = float(np.mean(times_opt))
speedup_mean = mean_orig / mean_opt
speedup_total = total_orig / total_opt

print("\nOriginal Implementation:")
print(f"  Total time:  {total_orig:.3f}s")
print(f"  Mean/freq:   {mean_orig:.3f}s")
print(f"  Min/freq:    {min(times_orig):.3f}s")
print(f"  Max/freq:    {max(times_orig):.3f}s")

print("\nOptimized Implementation:")
print(f"  Total time:  {total_opt:.3f}s")
print(f"  Mean/freq:   {mean_opt:.3f}s")
print(f"  Min/freq:    {min(times_opt):.3f}s")
print(f"  Max/freq:    {max(times_opt):.3f}s")

print(f"\n{'SPEEDUP':^30}")
print(f"  Per frequency: {speedup_mean:.2f}x")
print(f"  Total:         {speedup_total:.2f}x")
print(
    f"  Time saved:    {total_orig - total_opt:.2f}s ({100*(1-total_opt/total_orig):.1f}%)"
)

print(f"\n{'TIMING BREAKDOWN':^30}")
print(f"{'Operation':<20} {'Original':<12} {'Optimized':<12} {'Speedup':<10}")
print("-" * 55)
for op_name in ["t_init", "t_incident", "t_total"]:
    op_label = op_name.replace("t_", "").replace("_", " ").title()
    orig_mean = float(np.mean([r[op_name] for r in results_original]))
    opt_mean = float(np.mean([r[op_name] for r in results_optimized]))
    speedup = orig_mean / opt_mean if opt_mean > 0 else float("inf")
    print(f"{op_label:<20} {orig_mean:>10.3f}s  {opt_mean:>10.3f}s  {speedup:>8.2f}x")

# -----------------------------------------------------------------------------
# Extrapolation to full problem (adjust N if needed)
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("EXTRAPOLATION TO FULL PROBLEM")
print("=" * 70)

full_n_freq = 10  # adjust to your full run
full_time_orig = (total_orig / N_FREQ_TEST) * full_n_freq
full_time_opt = (total_opt / N_FREQ_TEST) * full_n_freq
full_time_saved = full_time_orig - full_time_opt

print(f"\nFor {full_n_freq} frequencies:")
print(f"  Original:     ~{full_time_orig:.1f}s ({full_time_orig/60:.1f} min)")
print(f"  Optimized:    ~{full_time_opt:.1f}s ({full_time_opt/60:.1f} min)")
print(f"  Time saved:   ~{full_time_saved:.1f}s ({full_time_saved/60:.1f} min)")
print(f"  Speedup:      {speedup_total:.2f}x")

# -----------------------------------------------------------------------------
# Final summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"{'Accuracy:':<12} {'VERIFIED' if all_passed else 'FAILED'}")
print(f"{'Speedup:':<12} {speedup_total:.2f}x faster on average")
print(f"{'GPU:':<12} {'Enabled' if GPU_AVAILABLE else 'Disabled'}")

print("\nKey optimizations assumed in optimized operator:")
print("  1. GPU memory caching (PML/V matrices)")
print("  2. Reduced CPU-GPU transfers")
print("  3. Pre-allocated workspace arrays")
print("  4. CUDA Graph/persistent sweep (depending on your implementation)")
print("\n" + "=" * 70)

sys.exit(0 if all_passed else 1)